# Pipeline v2 ChatGPT + Voiceover Workbench

Resume-first notebook for the manual production path after App 4.

Flow:

1. Load an existing `video_packaging_result.json` by `RESUME_RUN_ID`.
2. Export a ChatGPT `content_packet` handoff.
3. Paste the handoff into ChatGPT and save the returned narration script as text.
4. Run `src.pipeline_v2.voiceover` from that manual script text.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_V2 = PROJECT_ROOT / "output_v2"
PIPELINE_V2_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline_v2" / "config" / "workbench.yaml"
NEWS_COLLECTION_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "apps" / "news_collection"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root:  {OUTPUT_V2}")
print(f"Config:       {PIPELINE_V2_CONFIG_PATH}")


In [ ]:
# Set this to a run that has completed App 4: Video Packaging.
RESUME_RUN_ID = "v2_apps_2026_06_09_225504_daily_48h_all_sources"

# Paste the full sectioned ChatGPT response here. The notebook will parse the script block.
CHATGPT_RESPONSE_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_response.txt"
MANUAL_SCRIPT_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_full_script.txt"
PARSED_RESPONSE_PATH = OUTPUT_V2 / "manual_scripts" / RESUME_RUN_ID / "chatgpt_response_parsed.json"

WORKBENCH_CONFIG = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(WORKBENCH_CONFIG, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")

RUN_PROFILE = WORKBENCH_CONFIG.get("default_run_profile", "daily_48h_all_sources")
PROFILE_CONFIG = deepcopy(WORKBENCH_CONFIG["profiles"][RUN_PROFILE])

HANDOFF_OUTPUT_DIR = OUTPUT_V2 / "handoffs" / RESUME_RUN_ID
VOICEOVER_INPUT_DIR = OUTPUT_V2 / "voiceover_inputs"

print(f"Resume run:          {RESUME_RUN_ID}")
print(f"ChatGPT response:    {CHATGPT_RESPONSE_PATH}")
print(f"Parsed script path:  {MANUAL_SCRIPT_PATH}")
print(f"ChatGPT handoff dir: {HANDOFF_OUTPUT_DIR / 'chatgpt' / 'content_packet'}")


In [ ]:
def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return data


def load_optional_json(path: Path) -> dict:
    return load_json(path) if path.exists() else {}


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_V2": str(OUTPUT_V2),
        "RUN_ID": RESUME_RUN_ID,
    }
    if isinstance(value, str):
        return value.format(**context)
    if isinstance(value, list):
        return [render_config_templates(item) for item in value]
    if isinstance(value, dict):
        return {key: render_config_templates(item) for key, item in value.items()}
    return value


def voiceover_variant(config: dict) -> str:
    def slug(value: object) -> str:
        text = str(value).strip().lower().replace(".", "p")
        text = "".join(ch if ch.isalnum() else "_" for ch in text)
        return "_".join(part for part in text.split("_") if part)

    speed = config.get("speed")
    speed_label = f"speed_{slug(speed)}" if speed is not None else "style_speed"
    voice_label = slug(config.get("voice") or "default_voice")
    return "_".join(
        part for part in [slug(config.get("provider", "provider")), slug(config.get("style", "style")), voice_label, speed_label] if part
    )


def build_post_app4_manifest() -> dict:
    news_input = load_optional_json(OUTPUT_V2 / "news_collection" / RESUME_RUN_ID / "news_collection_input.json")
    evidence_input = load_optional_json(OUTPUT_V2 / "evidence_prep" / RESUME_RUN_ID / "evidence_prep_input.json")
    story_input = load_optional_json(OUTPUT_V2 / "story_mapping" / RESUME_RUN_ID / "story_mapping_input.json")
    video_input = load_optional_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_input.json")
    return {
        "run_id": RESUME_RUN_ID,
        "generated_at": datetime.now(IST).isoformat(),
        "pipeline_track": "pipeline_v2",
        "orchestrator": "notebooks/v2_chatgpt_voiceover_workbench.ipynb",
        "config_source": str(PIPELINE_V2_CONFIG_PATH.relative_to(PROJECT_ROOT)),
        "apps": {
            "news_collection": {"config": news_input.get("config", {})},
            "evidence_prep": {"config": evidence_input.get("config", {})},
            "story_mapping": {"config": story_input.get("config", {})},
            "video_packaging": {"config": video_input.get("config", {})},
        },
    }


def build_summary_bundle() -> dict:
    return {
        "news_collection": load_optional_json(OUTPUT_V2 / "news_collection" / RESUME_RUN_ID / "summary.json"),
        "evidence_prep": load_optional_json(OUTPUT_V2 / "evidence_prep" / RESUME_RUN_ID / "summary.json"),
        "story_mapping": load_optional_json(OUTPUT_V2 / "story_mapping" / RESUME_RUN_ID / "summary.json"),
        "video_packaging": load_optional_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "summary.json"),
    }

SECTION_MARKERS = ("NOTES", "SCRIPT", "METADATA")
PLACEHOLDER_RESPONSE_TEXT = "Paste the full CHATGPT_PIPELINE_RESPONSE_V1 response here, then rerun the parser cell.\n"


def parse_chatgpt_pipeline_response(text: str) -> dict:
    raw = text.strip()
    if not raw or raw.startswith("Paste the full CHATGPT_PIPELINE_RESPONSE_V1 response here"):
        raise ValueError("ChatGPT response file is empty or still a placeholder")
    if "CHATGPT_PIPELINE_RESPONSE_V1" not in raw[:500]:
        raise ValueError("ChatGPT response must start with or near CHATGPT_PIPELINE_RESPONSE_V1")

    marker_positions: list[tuple[int, str, int]] = []
    for marker in SECTION_MARKERS:
        token = f"[{marker}]"
        index = raw.find(token)
        if index == -1:
            raise ValueError(f"ChatGPT response is missing required section {token}")
        marker_positions.append((index, marker.lower(), index + len(token)))
    marker_positions.sort()

    sections = {}
    for position, name, content_start in marker_positions:
        next_positions = [item[0] for item in marker_positions if item[0] > position]
        content_end = min(next_positions) if next_positions else len(raw)
        sections[name] = raw[content_start:content_end].strip()

    script = clean_voiceover_script(sections["script"])
    if not script:
        raise ValueError("Parsed [SCRIPT] section is empty")
    warnings = validate_voiceover_script_text(script)
    return {
        "response_version": "CHATGPT_PIPELINE_RESPONSE_V1",
        "notes": sections.get("notes", ""),
        "script": script,
        "metadata_text": sections.get("metadata", ""),
        "warnings": warnings,
    }


def clean_voiceover_script(text: str) -> str:
    lines = []
    for line in text.replace("\r\n", "\n").replace("\r", "\n").split("\n"):
        cleaned = line.strip()
        if not cleaned:
            if lines and lines[-1] != "":
                lines.append("")
            continue
        if cleaned in {"---", "***"}:
            continue
        if cleaned.startswith("```"):
            continue
        # Remove common markdown styling while preserving the spoken sentence.
        if cleaned.startswith("**Transition:**"):
            cleaned = cleaned.replace("**Transition:**", "", 1).strip()
        elif cleaned.lower().startswith("transition:"):
            cleaned = cleaned.split(":", 1)[1].strip()
        cleaned = cleaned.replace("**", "").strip()
        if cleaned:
            lines.append(cleaned)
    return "\n".join(lines).strip()


def validate_voiceover_script_text(script: str) -> list[str]:
    warnings = []
    banned_prefixes = ("#", "##", "###", "target length:", "format:", "tone:")
    leaked_lines = [line for line in script.splitlines() if line.strip().lower().startswith(banned_prefixes)]
    if leaked_lines:
        warnings.append(f"Script may still contain headings/metadata lines: {leaked_lines[:5]}")
    if "[METADATA]" in script or "[NOTES]" in script or "CHATGPT_PIPELINE_RESPONSE_V1" in script:
        warnings.append("Script contains response wrapper markers")
    if "```" in script:
        warnings.append("Script contains markdown code fences")
    return warnings



## 1. Load App 4 Packet

This loads the existing `video_packaging_result.json`. Run this after setting `RESUME_RUN_ID` above.


In [ ]:
video_packaging_result = load_json(OUTPUT_V2 / "video_packaging" / RESUME_RUN_ID / "video_packaging_result.json")
if video_packaging_result.get("status") != "ok":
    raise RuntimeError(video_packaging_result.get("summary", {}))
if not video_packaging_result.get("handoff", {}).get("ready_for_script_generation", False):
    raise RuntimeError("video_packaging_result is not ready for script generation")

video_packet = video_packaging_result["data"]["video_packet"]
print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {len(video_packet.get('stories', []))}")
print(f"Target seconds:         {video_packet.get('target_duration_seconds')}")


## 2. Create ChatGPT Content Packet Handoff

Paste the generated files into ChatGPT in this order: `01_instructions.md`, `02_payload.md`, then optionally `03_manifest_context.md`.


In [ ]:
from src.pipeline_v2.handoffs.chatgpt_exporter import run as run_chatgpt_exporter

handoff_manifest = build_post_app4_manifest()
summary_bundle = build_summary_bundle()
chatgpt_output_dir = HANDOFF_OUTPUT_DIR / "chatgpt" / "content_packet"
chatgpt_handoff_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "profile": "content_packet",
        "prompt_version": "v1",
        "output_dir": str(chatgpt_output_dir),
    },
    "input": {
        "manifest": handoff_manifest,
        "artifacts": {
            "content_packet": video_packet,
        },
        "summaries": summary_bundle,
    },
}
chatgpt_handoff_result = run_chatgpt_exporter(chatgpt_handoff_input)
if chatgpt_handoff_result["status"] != "ok":
    raise RuntimeError(chatgpt_handoff_result["summary"])

chatgpt_paths = write_handoff_files(chatgpt_handoff_result["data"]["handoff_package"]["files"], chatgpt_output_dir)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_content_packet_handoff_input.json", chatgpt_handoff_input)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_content_packet_handoff_result.json", chatgpt_handoff_result)
write_json(HANDOFF_OUTPUT_DIR / "post_app4_manifest.json", handoff_manifest)
write_json(HANDOFF_OUTPUT_DIR / "post_app4_summary_bundle.json", summary_bundle)

print(f"ChatGPT handoff status: {chatgpt_handoff_result['status']}")
print(f"Files:                  {len(chatgpt_paths)}")
for path in chatgpt_paths:
    print(path)


## 3. Manual ChatGPT Step

Paste the entire ChatGPT response into `CHATGPT_RESPONSE_PATH`. The response should use `CHATGPT_PIPELINE_RESPONSE_V1` with `[NOTES]`, `[SCRIPT]`, and `[METADATA]` sections. The notebook will extract only `[SCRIPT]` for voiceover.

Suggested reminder prompt after pasting the handoff:

```text
Return your answer in CHATGPT_PIPELINE_RESPONSE_V1 format with [NOTES], [SCRIPT], and [METADATA].
The [SCRIPT] block must contain only spoken narration text, no markdown headings, no timing labels, no visual notes, and no citations list.
```


In [ ]:
CHATGPT_RESPONSE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not CHATGPT_RESPONSE_PATH.exists():
    CHATGPT_RESPONSE_PATH.write_text(PLACEHOLDER_RESPONSE_TEXT, encoding="utf-8")
    raise FileNotFoundError(f"Created placeholder response file. Fill it in, then rerun: {CHATGPT_RESPONSE_PATH}")
print(f"ChatGPT response file exists: {CHATGPT_RESPONSE_PATH}")


## 4. Parse ChatGPT Response

This extracts the `[SCRIPT]` block, writes `chatgpt_full_script.txt`, and saves notes/metadata/warnings to `chatgpt_response_parsed.json`.


In [ ]:
from src.pipeline_v2.voiceover.text_chunks import split_narration_into_chunks, word_count

chatgpt_response_text = CHATGPT_RESPONSE_PATH.read_text(encoding="utf-8")
parsed_chatgpt_response = parse_chatgpt_pipeline_response(chatgpt_response_text)
manual_script_text = parsed_chatgpt_response["script"]
manual_word_count = word_count(manual_script_text)

MANUAL_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
MANUAL_SCRIPT_PATH.write_text(manual_script_text + "\n", encoding="utf-8")
write_json(PARSED_RESPONSE_PATH, parsed_chatgpt_response)

print(f"Parsed script words: {manual_word_count}")
print(f"Parsed script path:  {MANUAL_SCRIPT_PATH}")
print(f"Parsed metadata:     {PARSED_RESPONSE_PATH}")
if parsed_chatgpt_response["warnings"]:
    print("Parser warnings:")
    for warning in parsed_chatgpt_response["warnings"]:
        print(f"- {warning}")


## 5. Run Voiceover From Parsed Script

This uses `input.script_text` from the parsed `[SCRIPT]` block, so it does not require App 5 `script_generation_result`.


In [ ]:
voiceover_config = render_config_templates(deepcopy(PROFILE_CONFIG.get("voiceover", {})))
if not voiceover_config:
    raise KeyError("Workbench profile is missing voiceover config")
voiceover_output_dir = OUTPUT_V2 / "voiceover" / f"{RESUME_RUN_ID}_{voiceover_variant(voiceover_config)}_chatgpt_script"
voiceover_config["output_dir"] = str(voiceover_output_dir)

voiceover_input = {
    "run": {"run_id": RESUME_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": voiceover_config,
    "input": {
        "script_text": manual_script_text,
        "script_source": str(MANUAL_SCRIPT_PATH),
    },
}

from src.pipeline_v2.voiceover.config import resolve_style
style_profile = resolve_style(voiceover_config["style"])
chunking = voiceover_config.get("chunking") or {}
resolved_chunking = {
    "max_chars": chunking.get("chunk_max_chars") or style_profile.chunk_max_chars,
    "max_sentences_per_chunk": chunking.get("max_sentences_per_chunk") or style_profile.max_sentences_per_chunk,
    "chunk_pause_seconds": (
        chunking.get("chunk_pause_seconds")
        if chunking.get("chunk_pause_seconds") is not None
        else style_profile.chunk_pause_seconds
    ),
    "paragraph_pause_seconds": (
        chunking.get("paragraph_pause_seconds")
        if chunking.get("paragraph_pause_seconds") is not None
        else style_profile.paragraph_pause_seconds
    ),
}
planned_chunks = split_narration_into_chunks(manual_script_text, **resolved_chunking)
if not planned_chunks:
    raise RuntimeError("Voiceover preflight produced zero chunks from manual script")

print(f"Voiceover provider:       {voiceover_input['config']['provider']}")
print(f"Voiceover style:          {voiceover_input['config']['style']}")
print(f"Voiceover speed:          {voiceover_input['config'].get('speed')}")
print(f"Voiceover output:         {voiceover_output_dir}")
print(f"Resolved chunking:        {resolved_chunking}")
print(f"Planned voiceover chunks: {len(planned_chunks)}")

VOICEOVER_INPUT_DIR.mkdir(parents=True, exist_ok=True)
voiceover_input_path = VOICEOVER_INPUT_DIR / f"{RESUME_RUN_ID}_{voiceover_variant(voiceover_input['config'])}_chatgpt_script_voiceover_input.json"
write_json(voiceover_input_path, voiceover_input)

voiceover_python = PROJECT_ROOT / ".venv-voiceover" / "bin" / "python"
if not voiceover_python.exists():
    raise FileNotFoundError(f"Voiceover venv Python not found: {voiceover_python}")

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [
        str(voiceover_python),
        "-m",
        "src.pipeline_v2.voiceover.cli",
        "--input-json",
        str(voiceover_input_path),
    ],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

voiceover_summary = load_json(voiceover_output_dir / "summary.json")
voiceover_package = load_json(voiceover_output_dir / "voiceover_package.json")
voiceover_result = {
    "status": "ok",
    "data": {
        "run_id": RESUME_RUN_ID,
        "generated_at": voiceover_package.get("generated_at"),
        "voiceover_package": voiceover_package,
    },
    "summary": voiceover_summary,
    "handoff": voiceover_package.get("handoff", {}),
}
write_json(voiceover_output_dir / "voiceover_input.json", voiceover_input)
write_json(voiceover_output_dir / "voiceover_result.json", voiceover_result)

print(f"Voiceover status: {voiceover_summary['status']}")
print(f"Voice:            {voiceover_summary['voice']}")
print(f"Duration seconds: {voiceover_summary['duration_seconds']}")
print(f"Chunks:           {voiceover_summary['chunk_count']}")
print(f"Audio path:       {voiceover_summary['audio_path']}")
print(f"Wrote:            {voiceover_output_dir}")


In [ ]:
voiceover_summary